## Homework 9: Text Classification with Fine-Tuned BERT

https://en.wikipedia.org/wiki/BERT_(language_model)

<font color = 'cyan'>Bi-directional encoder representations from transformers (BERT) is a language model introduced in October 2018 by researchers at Google. It learns to represent text as a sequence of vectors using self-supervised learning. It uses the encoder-only transformer architecture. BERT dramatically improved the state of the art for large language models. As of 2020, BERT is a ubiquitous baseline in natural language processing (NLP) experiments.</font>


### Due: Midnight on November 5th (with 2-hour grace period) — Worth 85 points

In this final homework, we’ll explore **fine-tuning a pre-trained Transformer model (BERT)** for text classification using the **IMDB Movie Review** dataset. You’ll begin with a working baseline notebook and then conduct a series of controlled experiments to understand how data size, context length, and model architecture affect performance.

You’ll complete three problems:

* **Problem 1:** Evaluate how **sequence length** and **learning rate** jointly influence validation loss and generalization.
* **Problem 2:** Measure how **training data size** affects both model performance and total training time.
* **Problem 3:** Compare **two additional models** from the BERT family to analyze the trade-offs between model size and accuracy on this dataset.

In each problem, you’ll report your key metrics, summarize what you observed, and reflect on what you learned.

> **Note:** This homework was developed and tested on **Google Colab**, due to version conflicts when running locally. It is **strongly recommended** that you complete your work on Colab as well.

There are 6 problems, each worth 14 points, and you get one point free if you complete the entire homework.


In [3]:
# Install once per new Colab runtime
%pip -q install -U keras keras-hub tensorflow tensorflow-text datasets evaluate

In [ ]:

import os
os.environ["KERAS_BACKEND"] = "tensorflow"

import time
import random
import numpy as np
import keras
import keras_hub as kh
import evaluate
import pandas as pd

from datasets import load_dataset, Dataset, Features, Value, ClassLabel

from keras import mixed_precision                    # generally faster
mixed_precision.set_global_policy("mixed_float16")

### Global hyperparameters for this homework

In [19]:
# ---------------- Config ----------------
SEED        = 42
MAX_LEN     = 128
EPOCHS      = 3
BATCH       = 32
EVAL_BATCH  = 64
SUBSET_FRAC = 0.25   # <-- 0.25 to train and test on 25% of whole dataset during development;  set to 1.0 for full dataset

keras.utils.set_random_seed(SEED)

### Load and Preprocess the IMDB Movie Review Dataset

#### load_and_split_data

In [106]:
def load_and_split_data(subset_frac, seed, count_func):
    """
    Loads the IMDB dataset, takes a stratified subset, and splits it into
    stratified 80/10/10 (Train/Val/Test) Numpy arrays.
    """
    print(f"Loading and splitting data (Subset: {subset_frac:.2f})...")

    start = time.time()

    # 1. Load IMDb (raw), join train+test
    imdb   = load_dataset("imdb")
    texts  = list(imdb["train"]["text"]) + list(imdb["test"]["text"])
    labels = np.array(list(imdb["train"]["label"]) + list(imdb["test"]["label"]), dtype="int32")

    # 2. Build DS with explicit features
    features = Features({"text": Value("string"),
                         "label": ClassLabel(num_classes=2, names=["NEG","POS"])})
    all_ds = Dataset.from_dict({"text": texts, "label": labels.tolist()}, features=features)

    # 3. Take a stratified subset of the FULL dataset based on subset_frac
    if 0.0 < subset_frac < 1.0:
        sub = all_ds.train_test_split(train_size=subset_frac, seed=seed, stratify_by_column="label")
        ds_pool = sub["train"]
    else:
        ds_pool = all_ds

    # 4. Stratified 80/10/10 split on the (possibly smaller) pool
    splits = ds_pool.train_test_split(test_size=0.20, seed=seed, stratify_by_column="label")
    train_val_pool, test_ds = splits["train"], splits["test"]
    splits2 = train_val_pool.train_test_split(test_size=0.125, seed=seed, stratify_by_column="label")
    train_ds, val_ds = splits2["train"], splits2["test"]

    # 5. Numpy arrays for Keras fit/predict
    X_tr = np.array(train_ds["text"], dtype=object); y_tr = np.array(train_ds["label"], dtype="int32")
    X_va = np.array(val_ds["text"],   dtype=object); y_va = np.array(val_ds["label"],   dtype="int32")
    X_te = np.array(test_ds["text"],  dtype=object); y_te = np.array(test_ds["label"],  dtype="int32")

    end           = time.time() - start
    elapsed_time  = time.strftime("%H:%M:%S", time.gmtime(end))

    print(f"Data loaded and split in {elapsed_time}")
    print(f"Train Size: {count_func(train_ds)[0]}, Val Size: {count_func(val_ds)[0]}, Test Size: {count_func(test_ds)[0]}")

    return X_tr, y_tr, X_va, y_va, X_te, y_te

In [107]:
# Helper function to count dataset labels
def _counts(ds):
    arr = np.array(ds["label"], dtype=int)
    return len(arr), np.bincount(arr, minlength=2).tolist()

In [93]:
X_tr, y_tr, X_va, y_va, X_te, y_te = load_and_split_data(SUBSET_FRAC, SEED, _counts)

Loading and splitting data (Subset: 0.25)...
Train Size: 8750, Val Size: 1250, Test Size: 2500


### Build and train a baseline Distil-Bert Text Classifier

In [17]:
# ---- Keras Hub preprocessor + classifier ----
preproc = kh.models.DistilBertTextClassifierPreprocessor.from_preset(
    "distil_bert_base_en_uncased",
    sequence_length = MAX_LEN
)
model = kh.models.DistilBertTextClassifier.from_preset(
    "distil_bert_base_en_uncased",
    num_classes   = 2,
    preprocessor  = preproc
)

model.compile(
    optimizer = keras.optimizers.Adam(1e-5),
    loss      = keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics   = [keras.metrics.SparseCategoricalAccuracy(name="acc")],
)

start = time.time()

# ---- Train with early stopping (restore best val weights) ----
cb = [keras.callbacks.EarlyStopping(monitor = "val_loss",
                                    patience = 2,
                                    restore_best_weights = True)]

history = model.fit(
    X_tr, y_tr,
    validation_data = (X_va, y_va),
    epochs          = EPOCHS,
    batch_size      = BATCH,
    callbacks       = cb,
    verbose         = 1,
)

# ---- Evaluate (accuracy + F1 via `evaluate`) ----
logits = model.predict(X_te, batch_size=EVAL_BATCH, verbose=0)
y_pred = logits.argmax(axis=-1)

acc_metric  = evaluate.load("accuracy")
f1_metric   = evaluate.load("f1")
acc         = acc_metric.compute(predictions = y_pred, references = y_te)["accuracy"]
f1          = f1_metric.compute(predictions = y_pred, references = y_te)["f1"]

# Tiny confusion matrix helper (no sklearn needed)
def confusion_matrix_np(y_true, y_pred, num_classes=2):
    cm = np.zeros((num_classes, num_classes), dtype=int)
    for t, p in zip(y_true, y_pred):
        cm[t, p] += 1
    return cm

print(f"\nValidation acc (best epoch): {history.history['val_acc'][np.argmin(history.history['val_loss'])]:.3f}")
print(f"\nTest accuracy: {acc:.3f}   Test F1: {f1:.3f}")
print("\nConfusion matrix:\n", confusion_matrix_np(y_te, y_pred))

end = time.time() - start
print("\nElapsed time:", time.strftime("%H:%M:%S", time.gmtime(end)))

Epoch 1/3
274/274 ━━━━━━━━━━━━━━━━━━━━ 83s 155ms/step - acc: 0.7779 - loss: 0.4565 - val_acc: 0.8440 - val_loss: 0.3584
Epoch 2/3
274/274 ━━━━━━━━━━━━━━━━━━━━ 10s 34ms/step - acc: 0.8827 - loss: 0.2860 - val_acc: 0.8544 - val_loss: 0.3484
Epoch 3/3
274/274 ━━━━━━━━━━━━━━━━━━━━ 10s 36ms/step - acc: 0.9183 - loss: 0.2149 - val_acc: 0.8600 - val_loss: 0.3581

Validation acc (best epoch): 0.854

Test accuracy: 0.852   Test F1: 0.855

Confusion matrix:
 [[1032  218]
 [ 153 1097]]

Elapsed time: 00:01:56


#### run_experiment

In [24]:
def run_experiment(max_len, lr, batch_size, name):

    print(f"\n Running: {name} (MAX_LEN={max_len}, LR={lr}, BATCH={batch_size}) ---")
    start = time.time()

    # ---- Keras Hub preprocessor + classifier ----
    preproc = kh.models.DistilBertTextClassifierPreprocessor.from_preset(
        "distil_bert_base_en_uncased",
        sequence_length = max_len
    )
    model = kh.models.DistilBertTextClassifier.from_preset(
        "distil_bert_base_en_uncased",
        num_classes   = 2,
        preprocessor  = preproc
    )

    model.compile(
        optimizer = keras.optimizers.Adam(lr),
        loss      = keras.losses.SparseCategoricalCrossentropy(from_logits = True),
        metrics   = [keras.metrics.SparseCategoricalAccuracy(name = "acc")],
    )


    # 3. Train with early stopping
    cb = [keras.callbacks.EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True)]

    history = model.fit(
        X_tr, y_tr,
        validation_data = (X_va, y_va),
        epochs          = EPOCHS,
        batch_size      = batch_size,
        callbacks       = cb,
        verbose         = 1,
    )

    # 4. Record Key Metric
    min_val_loss_idx  = np.argmin(history.history['val_loss'])
    best_val_acc      = history.history['val_acc'][min_val_loss_idx]

    # 5. Evaluate on Test set (for completeness)
    logits = model.predict(X_te, batch_size = EVAL_BATCH, verbose = 0)
    y_pred = logits.argmax(axis = -1)

    acc_metric  = evaluate.load("accuracy")
    f1_metric   = evaluate.load("f1")
    test_acc    = acc_metric.compute(predictions = y_pred, references = y_te)["accuracy"]
    test_f1     = f1_metric.compute(predictions = y_pred, references = y_te)["f1"]

    end           = time.time() - start
    elapsed_time  = time.strftime("%H:%M:%S", time.gmtime(end))

    print(f"-> Best Validation Accuracy: {best_val_acc:.4f} @ Epoch {min_val_loss_idx+1}")
    print(f"-> Test Accuracy: {test_acc:.4f}, Test F1: {test_f1:.4f}")
    print(f"-> Elapsed Time: {elapsed_time}")
    print("\nConfusion matrix:\n", confusion_matrix_np(y_te, y_pred))

    time_stamp = time.strftime("%Y-%m-%d_%H-%M-%S")

    return {
        "config_name":  name,
        "MAX_LEN":      max_len,
        "LR":           lr,
        "BATCH":        batch_size,
        "Best_Val_Acc": best_val_acc,
        "Test_Acc":     test_acc,
        "Test_F1":      test_f1,
        "elapsed_time": elapsed_time,
        'time_stamp':    time_stamp,
    }

#### run_experiment_p2

In [76]:
def run_experiment_p2(subset_frac, max_len, lr, batch_size):
    print(f"\n--- Running: SUBSET_FRAC={subset_frac:.2f} (Train/Pool Size: {int(50000 * subset_frac)}) ---")
    start = time.time()

    # 1. Load & split using helper function
    X_tr, y_tr, X_va, y_va, X_te, y_te = load_and_split_data(
        subset_frac, SEED, _counts
    )

    # 2. Model Setup and Training
    preproc = kh.models.DistilBertTextClassifierPreprocessor.from_preset(
        "distil_bert_base_en_uncased", sequence_length = max_len
    )
    model = kh.models.DistilBertTextClassifier.from_preset(
        "distil_bert_base_en_uncased", num_classes=2, preprocessor = preproc
    )

    model.compile(
        optimizer = keras.optimizers.Adam(lr),
        loss      = keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics   = [keras.metrics.SparseCategoricalAccuracy(name="acc")],
    )

    cb = [keras.callbacks.EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True)]
    history = model.fit(
        X_tr, y_tr,
        validation_data=(X_va, y_va),
        epochs=EPOCHS,
        batch_size=batch_size,
        callbacks=cb,
        verbose=1,
    )

    # 3. Evaluation
    min_val_loss_idx  = np.argmin(history.history['val_loss'])
    best_val_acc      = history.history['val_acc'][min_val_loss_idx]

    logits            = model.predict(X_te, batch_size=EVAL_BATCH, verbose=0)
    y_pred            = logits.argmax(axis=-1)

    acc_metric        = evaluate.load("accuracy")
    f1_metric         = evaluate.load("f1")
    test_acc          = acc_metric.compute(predictions=y_pred, references=y_te)["accuracy"]
    test_f1           = f1_metric.compute(predictions=y_pred, references=y_te)["f1"]

    end           = time.time() - start
    elapsed_time  = time.strftime("%H:%M:%S", time.gmtime(end))
    time_stamp    = time.strftime("%Y-%m-%d_%H-%M-%S")

    print(f"-> Best Validation Accuracy: {best_val_acc:.4f}")
    print(f"-> Test Accuracy: {test_acc:.4f}, Test F1: {test_f1:.4f}")
    print(f"-> Elapsed Time: {elapsed_time}")

    return {
        "SUBSET_FRAC":  subset_frac,
        # This will now use the size of the returned train_ds (which is local to the data function)
        "MAX_LEN":      max_len,
        "LR":           lr,
        "BATCH":        batch_size,
        "Train_Size":   len(y_tr),
        "Best_Val_Acc": best_val_acc,
        "Test_Acc":     test_acc,
        "Test_F1":      test_f1,
        "elapsed_time": elapsed_time,
        'time_stamp':   time_stamp,
    }

#### run_experiment_p3

In [105]:
def run_experiment_p3(model_name, preproc_cls, model_cls, preset, max_len, lr, batch_size):
    print(f"\n--- Running Model: {model_name} ---")

    start = time.time()

    # 1. Model Setup (Preproc and Classifier based on parameters)
    preproc = preproc_cls.from_preset(preset, sequence_length = max_len)
    model   = model_cls.from_preset(preset, num_classes=2, preprocessor=preproc)

    # 2. Compile (Fixed LR)
    model.compile(
        optimizer = keras.optimizers.Adam(lr),
        loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics   = [keras.metrics.SparseCategoricalAccuracy(name="acc")],
    )

    # 3. Train
    cb = [keras.callbacks.EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True)]
    history = model.fit(
        X_tr, y_tr,
        validation_data=(X_va, y_va),
        epochs=EPOCHS,
        batch_size = batch_size,
        callbacks=cb,
        verbose=1,
    )

    # 4. Evaluation
    min_val_loss_idx  = np.argmin(history.history['val_loss'])
    best_val_acc      = history.history['val_acc'][min_val_loss_idx]

    logits            = model.predict(X_te, batch_size = EVAL_BATCH, verbose=0)
    y_pred            = logits.argmax(axis=-1)

    acc_metric        = evaluate.load("accuracy")
    f1_metric         = evaluate.load("f1")
    test_acc          = acc_metric.compute(predictions=y_pred, references=y_te)["accuracy"]
    test_f1           = f1_metric.compute(predictions=y_pred, references=y_te)["f1"]

    end               = time.time() - start
    elapsed_time      = time.strftime("%H:%M:%S", time.gmtime(end))
    time_stamp        = time.strftime("%Y-%m-%d_%H-%M-%S")

    print(f"-> Best Validation Acc: {best_val_acc:.4f}")
    print(f"-> Test Acc: {test_acc:.4f}, Test F1: {test_f1:.4f}")
    print(f"-> Elapsed Time: {elapsed_time}")

    return {
        "Model":        model_name,
        "Best_Val_Acc": best_val_acc,
        "Test_Acc":     test_acc,
        "Test_F1":      test_f1,
        "elapsed_time":         elapsed_time,
        'time_stamp':   time_stamp,
    }

# Problem 1 — Mini sweep: context length × learning rate (6 runs)

In this problem we'll see how much **context length** (`MAX_LEN`) helps, and how sensitive fine-tuning is to **learning rate**—without running a huge grid.

## Setup (keep these fixed)

* `SUBSET_FRAC = 0.25`               # use only this percentage of the whole dataset
* `EPOCHS = 3`
* `BATCH = 32` (but see note for 256 below)
* **EarlyStopping** with `restore_best_weights=True`
* Same random `SEED` for all runs
* Same data split for all runs (don’t reshuffle between runs)

### Run these 6 configurations

**For each** `MAX_LEN ∈ {128, 256, 512}`, try **two** learning rates:

* **MAX_LEN = 128**

  * `(LR = 2e-5, BATCH = 32)` – healthy default for shorter contexts.
  * `(LR = 1e-5, BATCH = 32)` – conservative LR; often a touch stabler.

* **MAX_LEN = 256**

  * `(LR = 1e-5, BATCH = 16)` – longer context → lower batch.
  * `(LR = 7.5e-6, BATCH = 16)` – even steadier if loss is noisy.

* **MAX_LEN = 512**  *(heavier quadratic attention cost)*

  * `(LR = 7.5e-6, BATCH = 8)` – safe starting point.
  * `(LR = 5e-6, BATCH = 8)` – extra caution for stability.

**If you hit an Out Of Memory error:**

* At **256** with `BATCH = 16`, drop to `BATCH = 8`.
* At **512** with `BATCH = 8`, drop to `BATCH = 4`.


Then answer the graded questions.


In [25]:
# Your code here; add as many cells as you need


configurations = [
    {"max_len": 128,
     "lr": 2e-5,
     "batch": 32,
     "name": "128_2e-5_32"},

    {"max_len": 128,
     "lr": 1e-5,
     "batch": 32,
     "name": "128_1e-5_32"},

    {"max_len": 256,
     "lr": 1e-5,
     "batch": 16,
     "name": "256_1e-5_16"},

    {"max_len": 256,
     "lr": 7.5e-6,
     "batch": 16,
     "name": "256_7.5e-6_16"},

    {"max_len": 512, "lr": 7.5e-6, "batch": 8, "name": "512_7.5e-6_8"},
    {"max_len": 512, "lr": 5e-6, "batch": 8, "name": "512_5e-6_8"},
]


In [26]:
results = []

for config in configurations:
    result = run_experiment(config["max_len"], config["lr"], config["batch"], config["name"])
    results.append(result)


 Running: 128_2e-5_32 (MAX_LEN=128, LR=2e-05, BATCH=32) ---
Epoch 1/3
274/274 ━━━━━━━━━━━━━━━━━━━━ 85s 159ms/step - acc: 0.8075 - loss: 0.4106 - val_acc: 0.8408 - val_loss: 0.3531
Epoch 2/3
274/274 ━━━━━━━━━━━━━━━━━━━━ 10s 35ms/step - acc: 0.8955 - loss: 0.2548 - val_acc: 0.8464 - val_loss: 0.3561
Epoch 3/3
274/274 ━━━━━━━━━━━━━━━━━━━━ 11s 40ms/step - acc: 0.9313 - loss: 0.1781 - val_acc: 0.8600 - val_loss: 0.3764
-> Best Validation Accuracy: 0.8408 @ Epoch 1
-> Test Accuracy: 0.8424, Test F1: 0.8531
-> Elapsed Time: 00:02:20

Confusion matrix:
 [[ 962  288]
 [ 106 1144]]

 Running: 128_1e-5_32 (MAX_LEN=128, LR=1e-05, BATCH=32) ---
Epoch 1/3
274/274 ━━━━━━━━━━━━━━━━━━━━ 87s 164ms/step - acc: 0.7784 - loss: 0.4565 - val_acc: 0.8432 - val_loss: 0.3583
Epoch 2/3
274/274 ━━━━━━━━━━━━━━━━━━━━ 10s 37ms/step - acc: 0.8829 - loss: 0.2860 - val_acc: 0.8536 - val_loss: 0.3487
Epoch 3/3
274/274 ━━━━━━━━━━━━━━━━━━━━ 12s 42ms/step - acc: 0.9182 - loss: 0.2148 - val_acc: 0.8600 - val_loss: 0.3584
-

In [42]:
results_df_p1 = pd.DataFrame(results)
results_df_p1

,config_name,MAX_LEN,LR,BATCH,Best_Val_Acc,Test_Acc,Test_F1,elapsed_time,time_stamp
0,128_2e-5_32,128,0.000020,32,0.8408,0.8424,0.853095,00:02:20,2025-10-30_20-43-45
1,128_1e-5_32,128,0.000010,32,0.8536,0.8516,0.855473,00:02:22,2025-10-30_20-46-07
2,256_1e-5_16,256,0.000010,16,0.9064,0.8912,0.892320,00:03:31,2025-10-30_20-49-39
3,256_7.5e-6_16,256,0.000008,16,0.9032,0.8920,0.893196,00:02:45,2025-10-30_20-52-24
4,512_7.5e-6_8,512,0.000008,8,0.9136,0.9144,0.915549,00:04:25,2025-10-30_20-56-50
5,512_5e-6_8,512,0.000005,8,0.9072,0.9072,0.907274,00:03:31,2025-10-30_21-00-22


## Graded Questions

In [54]:
print(results_df_p1.index.is_unique)

True


In [57]:
# results_df_p1 = results_df_p1.reset_index(drop = True)
criteria      = results_df_p1['Best_Val_Acc'].idxmax()
best_run_df   = results_df_p1.loc[[criteria]]
best_run_df


,config_name,MAX_LEN,LR,BATCH,Best_Val_Acc,Test_Acc,Test_F1,elapsed_time,time_stamp
4,512_7.5e-6_8,512,0.000008,8,0.9136,0.9144,0.915549,00:04:25,2025-10-30_20-56-50


### Question a1a


In [66]:
# Set a1a to the validation accuracy at min validation loss for your best configuration found in this problem


a1a = best_run_df['Best_Val_Acc'].values[0]          # Replace 0.0 with your answer

In [67]:
# Graded Answer
# DO NOT change this cell in any way

print(f'a1a = {a1a:.4f}')

a1a = 0.9136


### Question a1b:



#### Your answers here...

*Does **more context** (128 → 256 → 512) consistently help?*

  <font color = 'plum'> Yes, more context consistently helps, at least with this IMDB dataset.

 <font color = 'plum'>

- Validation accuracy at minimum loss jumped about 5% from 0.8536 to 0.9064 when increasing context from 128 to 256 tokens-- I can surmise that a substantial amount of sentiment information for classification lies outside the first 128 tokens of the movie reviews. </font>

- When goign from 256 to 512 tokens, the relative gain was a lot smaller (best validation accuracy ~0.9136), but this came at sigificant cost in training time from 3:30 to 4:25. </font>


*How much effect did the learning rate have on the validation accuracy?*

<font color = 'plum'> Learning Rate had an inconsistent effect on validation accuracy.

* At `max_len = 128`, the effect was most pronounced: lower LR of $1 \times 10^{-5}$ performed better ($0.8536$) than $2 \times 10^{-5}$ ($0.8408$). Maybe the higher rate caused the optimizer to overshoot the optimum or made the training less stable.</font>

<font color = 'plum'>

* At `max_len = 512`, the effect was smaller (less than $1\%$ difference), but in both cases, the slightly higher LR in the pair performed better ($1 \times 10^{-5}$ over $7.5 \times 10^{-6}$ for 256, and $7.5 \times 10^{-6}$ over $5 \times 10^{-6}$ for 512).</font>


# Problem 2 — How much data is enough?

In this problem, you’ll investigate how model performance scales with dataset size.

**Setup.**
Use the best `MAX_LEN` and `LR` values you found in **Problem 1**.

**What to do:**

1. For each value of `SUBSET_FRAC ∈ {0.25, 0.50, 0.75, 1.00}`, train your model once and observe the displayed performance metrics.
2. Answer the discussion question below.




In [77]:
# Your code here; add as many cells as you need

best_max_len  = best_run_df['MAX_LEN'].values[0]
best_lr       = best_run_df['LR'].values[0]
best_batch    = best_run_df['BATCH'].values[0]

print(f"best_max_len: {best_max_len}, best_lr: {best_lr}, best_batch: {best_batch}")

best_max_len: 512, best_lr: 7.5e-06, best_batch: 8


In [78]:
subset_fractions = [0.25, 0.50, 0.75, 1.0]

results_p2 = []

for fraction in subset_fractions:
    result = run_experiment_p2(fraction, best_max_len, best_lr, best_batch)
    results_p2.append(result)

results_p2_df = pd.DataFrame(results_p2)



--- Running: SUBSET_FRAC=0.25 (Train/Pool Size: 12500) ---
Train Size: 8750, Val Size: 1250, Test Size: 2500
Epoch 1/3
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 110s 60ms/step - acc: 0.8573 - loss: 0.3190 - val_acc: 0.9192 - val_loss: 0.2175
Epoch 2/3
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 33s 30ms/step - acc: 0.9319 - loss: 0.1783 - val_acc: 0.9208 - val_loss: 0.2342
Epoch 3/3
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 33s 30ms/step - acc: 0.9607 - loss: 0.1141 - val_acc: 0.9184 - val_loss: 0.2176
-> Best Validation Accuracy: 0.9192
-> Test Accuracy: 0.9092, Test F1: 0.9109
-> Elapsed Time: 00:03:47

--- Running: SUBSET_FRAC=0.50 (Train/Pool Size: 25000) ---
Train Size: 17500, Val Size: 2500, Test Size: 5000
Epoch 1/3
2188/2188 ━━━━━━━━━━━━━━━━━━━━ 150s 49ms/step - acc: 0.8810 - loss: 0.2795 - val_acc: 0.9224 - val_loss: 0.1889
Epoch 2/3
2188/2188 ━━━━━━━━━━━━━━━━━━━━ 64s 29ms/step - acc: 0.9373 - loss: 0.1688 - val_acc: 0.9228 - val_loss: 0.2040
Epoch 3/3
2188/2188 ━━━━━━━━━━━━━━━━━━━━ 64s 29ms/step - acc: 0.9601

In [79]:
results_p2_df

,SUBSET_FRAC,MAX_LEN,LR,BATCH,Train_Size,Best_Val_Acc,Test_Acc,Test_F1,elapsed_time,time_stamp
0,0.25,512,0.000008,8,8750,0.919200,0.909200,0.910876,00:03:47,2025-10-30_23-21-08
1,0.50,512,0.000008,8,17500,0.922400,0.920200,0.918654,00:05:31,2025-10-30_23-26-40
2,0.75,512,0.000008,8,26250,0.921867,0.921067,0.922269,00:07:26,2025-10-30_23-34-06
3,1.00,512,0.000008,8,35000,0.932400,0.928400,0.929666,00:08:06,2025-10-30_23-42-13


In [83]:
criteria_p2       = results_p2_df['Best_Val_Acc'].idxmax()
best_run_p2_df    = results_p2_df.loc[[criteria_p2]]
best_run_p2_df

,SUBSET_FRAC,MAX_LEN,LR,BATCH,Train_Size,Best_Val_Acc,Test_Acc,Test_F1,elapsed_time,time_stamp
3,1.0,512,0.000008,8,35000,0.9324,0.9284,0.929666,00:08:06,2025-10-30_23-42-13


## Graded Questions

### Question a2a

In [84]:
# Set a2a to the validation accuracy at min validation loss for your best configuration found in this problem
# (Yes, it is probably at 1.0!)

a2a = best_run_p2_df['Best_Val_Acc'].values[0]            # Replace 0.0 with your answer

In [85]:
# Graded Answer
# DO NOT change this cell in any way

print(f'a2a = {a2a:.4f}')

a2a = 0.9324


### Question a2b:


1. *Summarize what you observed as dataset size increased.*

2. *Given that validation metrics are typically reliable to only about two decimal places, do the performance gains justify using the entire dataset?*

3. *What trade-offs between accuracy and computation time did you notice?*

#### Your Answer Here:


<font color = 'plum'>

- There was a consistent but modest non-linear increase in Best Validation Accuracy and Test Accuracy as the training dataset size (SUBSET_FRAC) increased from $25\%$ to $100\%$. </font>

<font color = 'plum'>

* I suppose the performance gains justify using the whole dataset. The final step from 0.75 to 1.00 boosted validation accuracy by 1.05%, likely due to variance in the early stopping epoch... Slightly over 1% for an extra 40 seconds of computation time, so one could argue that training the whole dataset was worth it. </font>

<font color = 'plum'>

- It's expected that training time will scale linearly with the number of samples, but that total time includes overheads which might explain why the final 25% only added 40 seconds in training time. However, I paid a penalty of +3 min 39 seconds for a puny gain in accuracy of 0.0027 by increasing the dataset size from 0.25 to 0.75. </font>

<font color = 'plum'>

- BUT, from 0.75 to 1.00, the accelrating  accuracy gain was worth the extra 40 seconds in comp time. So, while the ROI in terms of validation accuracy improvement per minute of additional training for this dataset might trend upward, it's definitely not linear growth across the training set; the growth comes in fits and bursts. </font>

# Problem 3 — Model swap: speed vs. accuracy (why: capacity matters)

In this problem we will compare encoder-only backbones of different sizes.

**Setup.** Keep the best `MAX_LEN`, `LR`, and `SUBSET_FRAC` from Problems 1–2. Only change the model/preset:

* **DistilBERT** (current baseline)
* **MobileBERT** (smaller/faster)
* **BERT-base** (larger/usually stronger)

**How to switch (two lines each).**

* DistilBERT:

  ```python
  preproc = kh.models.DistilBertTextClassifierPreprocessor.from_preset("distil_bert_base_en_uncased", sequence_length=MAX_LEN)
  model  = kh.models.DistilBertTextClassifier.from_preset("distil_bert_base_en_uncased", num_classes=2, preprocessor=preproc)
  ```
* MobileBERT:

  ```python
  preproc = kh.models.MobileBertTextClassifierPreprocessor.from_preset("mobile_bert_en_uncased", sequence_length=MAX_LEN)
  model  = kh.models.MobileBertTextClassifier.from_preset("mobile_bert_en_uncased", num_classes=2, preprocessor=preproc)
  ```
* BERT-base:

  ```python
  preproc = kh.models.BertTextClassifierPreprocessor.from_preset("bert_base_en_uncased", sequence_length=MAX_LEN)
  model  = kh.models.BertTextClassifier.from_preset("bert_base_en_uncased", num_classes=2, preprocessor=preproc)
  ```

**What to do.**

1. Train/evaluate each model once with identical settings.
2. Observe the performance metrics for each.
3. Answer the graded questions.



In [112]:
best_max_len_p2  = best_run_p2_df['MAX_LEN'].values[0]
best_lr_p2       = best_run_p2_df['LR'].values[0]
best_batch_p2    = best_run_p2_df['BATCH'].values[0]

SUBSET_FRAC_P3 = 1.00

print(f'best_max_len_p2: {best_max_len_p2}\nbest_lr_p2:{best_lr_p2}\nbest_batch_p2:{best_batch_p2}')

best_max_len_p2: 512
best_lr_p2:7.5e-06
best_batch_p2:8


In [109]:
# Your code here; add as many cells as you wish

X_tr, y_tr, X_va, y_va, X_te, y_te = load_and_split_data(SUBSET_FRAC_P3, SEED, _counts)

Loading and splitting data (Subset: 1.00)...
Data loaded and split in 00:00:18
Train Size: 35000, Val Size: 5000, Test Size: 10000


In [113]:
models_to_run = [
    {
        "name":        "DistilBERT",
        "preproc_cls": kh.models.DistilBertTextClassifierPreprocessor,
        "model_cls":   kh.models.DistilBertTextClassifier,
        "preset":      "distil_bert_base_en_uncased"
    },

    {
        "name":         "BERT-base",
        "preproc_cls":  kh.models.BertTextClassifierPreprocessor,
        "model_cls":    kh.models.BertTextClassifier,
        "preset":       "bert_base_en_uncased"
    }
]

In [114]:
results_p3 = []

for model in models_to_run:
    result = run_experiment_p3(
        model["name"],
        model["preproc_cls"],
        model["model_cls"],
        model["preset"],
        best_max_len_p2,
        best_lr_p2,
        best_batch_p2
    )

    results_p3.append(result)




--- Running Model: DistilBERT ---
Epoch 1/3
4375/4375 ━━━━━━━━━━━━━━━━━━━━ 177s 30ms/step - acc: 0.8983 - loss: 0.2491 - val_acc: 0.9322 - val_loss: 0.1772
Epoch 2/3
4375/4375 ━━━━━━━━━━━━━━━━━━━━ 127s 29ms/step - acc: 0.9424 - loss: 0.1550 - val_acc: 0.9304 - val_loss: 0.1947
Epoch 3/3
4375/4375 ━━━━━━━━━━━━━━━━━━━━ 127s 29ms/step - acc: 0.9651 - loss: 0.1028 - val_acc: 0.9282 - val_loss: 0.2260
-> Best Validation Acc: 0.9322
-> Test Acc: 0.9254, Test F1: 0.9265
-> Elapsed Time: 00:07:50

--- Running Model: BERT-base ---
Epoch 1/3
4375/4375 ━━━━━━━━━━━━━━━━━━━━ 336s 60ms/step - acc: 0.9123 - loss: 0.2227 - val_acc: 0.9382 - val_loss: 0.1655
Epoch 2/3
4375/4375 ━━━━━━━━━━━━━━━━━━━━ 254s 58ms/step - acc: 0.9591 - loss: 0.1185 - val_acc: 0.9478 - val_loss: 0.1647
Epoch 3/3
4375/4375 ━━━━━━━━━━━━━━━━━━━━ 254s 58ms/step - acc: 0.9793 - loss: 0.0660 - val_acc: 0.9446 - val_loss: 0.1931
-> Best Validation Acc: 0.9478
-> Test Acc: 0.9423, Test F1: 0.9425
-> Elapsed Time: 00:14:57


In [115]:
results_p3_df = pd.DataFrame(results_p3)
results_p3_df

,Model,Best_Val_Acc,Test_Acc,Test_F1,elapsed_time,time_stamp
0,DistilBERT,0.9322,0.9254,0.926473,00:07:50,2025-10-31_01-49-28
1,BERT-base,0.9478,0.9423,0.942455,00:14:57,2025-10-31_02-04-26


In [ ]:
# print(results_p3_df.to_markdown(index=False))

| Model      |   Best_Val_Acc |   Test_Acc |   Test_F1 | elapsed_time   | time_stamp          |
|:-----------|---------------:|-----------:|----------:|:---------------|:--------------------|
| DistilBERT |         0.9322 |     0.9254 |  0.926473 | 00:07:50       | 2025-10-31_01-49-28 |
| BERT-base  |         0.9478 |     0.9423 |  0.942455 | 00:14:57       | 2025-10-31_02-04-26 |


In [116]:
criteria_p3       = results_p3_df['Best_Val_Acc'].idxmax()
best_run_p3_df    = results_p3_df.loc[[criteria_p3]]
best_run_p3_df

,Model,Best_Val_Acc,Test_Acc,Test_F1,elapsed_time,time_stamp
1,BERT-base,0.9478,0.9423,0.942455,00:14:57,2025-10-31_02-04-26


## Graded Questions

### Question a3a

In [117]:
# Set a1a to the validation accuracy at min validation loss for your best model found in this problem

a3a = best_run_p3_df['Best_Val_Acc'].values[0]             # Replace 0.0 with your answer

In [118]:
# Graded Answer
# DO NOT change this cell in any way

print(f'a3a = {a3a:.4f}')

a3a = 0.9478


### Question a3b:


**Answer briefly.**

* Which model gives the best **accuracy/F1**?
* Which is **fastest** per epoch?
* Given limited development time or compute resources, which model is the best **overall choice** and why?




#### Your Answer Here:

<font color='plum'>

- BERT-base model, with larger size and capacity, delivered the best accuracy and F1 score of  0.9478 and 0.9425 respectively, a significant improvement of 1.5\% over DistilBERT. </font>

<font color='plum'>


- Clearly  DistilBERT is the fastest per epoch,  completing the full training run in 470 seconds compared to  897 seconds for BERT-base under these high-context (MAX_LEN = 512) conditions. </font>

<font color='plum'>


- DistilBERT is the best choice under time and compute constraints. While BERT-base provides the highest absolute accuracy, the gain is relatively small ($1.5\%$ absolute) compared to the nearly $2\times$ increase in training time and computational cost; meanwhile, DistilBERT is still highly accurate ($>93\%$) and nearly twice as efficien. </font>